# Etapa 1 — Carregamento e Preparação dos Dados

**Objetivo:** Carregar o dataset Store Sales, filtrar a família BEVERAGES, agregar por data e enriquecer com variáveis exógenas (preço do petróleo, feriados nacionais, onpromotion).

**Saída:** `data/processed/beverages_daily.csv`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

DATA_DIR = '../data/'
PROCESSED_DIR = '../data/processed/'

print('Bibliotecas carregadas.')

Bibliotecas carregadas.


## 1.1 Carregar train.csv e filtrar BEVERAGES

In [2]:
print('Carregando train.csv...')
train = pd.read_csv(DATA_DIR + 'train.csv', parse_dates=['date'])
print(f'  train.csv: {train.shape[0]:,} linhas, {train.shape[1]} colunas')
print(f'  Colunas: {list(train.columns)}')
print(f'  Período: {train.date.min().date()} a {train.date.max().date()}')

Carregando train.csv...


  train.csv: 3,000,888 linhas, 6 colunas
  Colunas: ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion']
  Período: 2013-01-01 a 2017-08-15


In [3]:
beverages = train[train['family'] == 'BEVERAGES'].copy()
print(f'Após filtro BEVERAGES: {beverages.shape[0]:,} linhas')
print(f'Lojas únicas: {beverages.store_nbr.nunique()}')
print(f'Datas únicas: {beverages.date.nunique()}')

Após filtro BEVERAGES: 90,936 linhas
Lojas únicas: 54
Datas únicas: 1684


## 1.2 Agregar por data (série diária da rede inteira)

In [4]:
daily = beverages.groupby('date').agg(
    sales=('sales', 'sum'),
    onpromotion=('onpromotion', 'sum')
).reset_index()

daily = daily.sort_values('date').reset_index(drop=True)
print(f'Série diária agregada: {len(daily)} observações')
print(f'Período: {daily.date.min().date()} a {daily.date.max().date()}')
daily.head()

Série diária agregada: 1684 observações
Período: 2013-01-01 a 2017-08-15


,date,sales,onpromotion
0,2013-01-01,810.0,0
1,2013-01-02,72092.0,0
2,2013-01-03,52105.0,0
3,2013-01-04,54167.0,0
4,2013-01-05,77818.0,0


## 1.3 Verificação de integridade

In [5]:
# Verificar continuidade das datas
expected_dates = pd.date_range(start=daily.date.min(), end=daily.date.max(), freq='D')
missing_dates = expected_dates.difference(daily.date)
print(f'Datas esperadas: {len(expected_dates)}')
print(f'Datas presentes: {len(daily)}')
print(f'Datas ausentes: {len(missing_dates)}')
if len(missing_dates) > 0:
    print(f'  → {missing_dates.tolist()}')

Datas esperadas: 1688
Datas presentes: 1684
Datas ausentes: 4
  → [Timestamp('2013-12-25 00:00:00'), Timestamp('2014-12-25 00:00:00'), Timestamp('2015-12-25 00:00:00'), Timestamp('2016-12-25 00:00:00')]


In [6]:
# Verificar zeros
zeros = daily[daily['sales'] == 0]
print(f'Zeros na série de vendas: {len(zeros)}')
if len(zeros) > 0:
    print('Datas com zero:')
    for _, row in zeros.iterrows():
        print(f'  {row.date.date()} — dia da semana: {row.date.day_name()}')

Zeros na série de vendas: 0


In [7]:
# Decisão: manter zeros (representam fechamento real das lojas em 1º de janeiro)
# Ver DECISOES.md — entrada sobre tratamento dos zeros
print('Zeros mantidos na série (fechamento real em 1º de janeiro).')
print(f'Estatísticas básicas:')
print(daily['sales'].describe())

Zeros mantidos na série (fechamento real em 1º de janeiro).
Estatísticas básicas:
count      1684.000000
mean     128832.830166
std       63121.645563
min         810.000000
25%       69815.750000
50%      128207.500000
75%      170349.500000
max      339352.000000
Name: sales, dtype: float64


## 1.4 Enriquecimento com preço do petróleo

In [8]:
oil = pd.read_csv(DATA_DIR + 'oil.csv', parse_dates=['date'])
oil = oil.rename(columns={'dcoilwtico': 'oil_price'})
print(f'oil.csv: {len(oil)} linhas')
print(f'NaN no preço do petróleo: {oil.oil_price.isna().sum()}')
print(f'Período: {oil.date.min().date()} a {oil.date.max().date()}')

oil.csv: 1218 linhas
NaN no preço do petróleo: 43
Período: 2013-01-01 a 2017-08-31


In [9]:
# Garantir cobertura completa do período da série diária com forward-fill
full_dates = pd.DataFrame({'date': expected_dates})
oil_full = full_dates.merge(oil, on='date', how='left')
oil_full['oil_price'] = oil_full['oil_price'].ffill()

# Verificar se ainda há NaN (pode ocorrer no início da série se não houver dado anterior)
nan_remaining = oil_full['oil_price'].isna().sum()
if nan_remaining > 0:
    # Backward fill para datas no início da série sem dado anterior
    oil_full['oil_price'] = oil_full['oil_price'].bfill()
    print(f'NaN restantes após forward-fill: {nan_remaining} — aplicado backward-fill nas datas iniciais.')
else:
    print('Forward-fill aplicado. Nenhum NaN restante no preço do petróleo.')

# Merge com série diária
daily = daily.merge(oil_full[['date', 'oil_price']], on='date', how='left')
print(f'NaN em oil_price após merge: {daily.oil_price.isna().sum()}')

NaN restantes após forward-fill: 1 — aplicado backward-fill nas datas iniciais.
NaN em oil_price após merge: 0


## 1.5 Enriquecimento com feriados nacionais

In [10]:
holidays = pd.read_csv(DATA_DIR + 'holidays_events.csv', parse_dates=['date'])
print(f'holidays_events.csv: {len(holidays)} linhas')
print(f'Tipos: {holidays.type.value_counts().to_dict()}')
print(f'Locales: {holidays.locale.value_counts().to_dict()}')
print(f'Transferred únicos: {holidays.transferred.unique()}')

holidays_events.csv: 350 linhas
Tipos: {'Holiday': 221, 'Event': 56, 'Additional': 51, 'Transfer': 12, 'Bridge': 5, 'Work Day': 5}
Locales: {'National': 174, 'Local': 152, 'Regional': 24}
Transferred únicos: [False  True]


In [11]:
# Filtrar feriados nacionais não transferidos
national_holidays = holidays[
    (holidays['locale'] == 'National') &
    (holidays['transferred'] == False)
][['date']].drop_duplicates()

national_holidays['is_national_holiday'] = 1
print(f'Feriados nacionais não transferidos: {len(national_holidays)}')
print(national_holidays.sort_values('date').head(10))

Feriados nacionais não transferidos: 160
         date  is_national_holiday
14 2012-08-10                    1
20 2012-10-12                    1
21 2012-11-02                    1
22 2012-11-03                    1
31 2012-12-21                    1
33 2012-12-22                    1
34 2012-12-23                    1
35 2012-12-24                    1
37 2012-12-25                    1
38 2012-12-26                    1


In [12]:
daily = daily.merge(national_holidays[['date', 'is_national_holiday']], on='date', how='left')
daily['is_national_holiday'] = daily['is_national_holiday'].fillna(0).astype(int)
print(f'Dias com feriado nacional na série: {daily.is_national_holiday.sum()}')

Dias com feriado nacional na série: 136


## 1.6 Verificação final e salvamento

In [13]:
print('=== Verificação Final ===')
print(f'Shape: {daily.shape}')
print(f'Colunas: {list(daily.columns)}')
print(f'NaN por coluna:')
print(daily.isna().sum())
print(f'\nPrimeiras linhas:')
display(daily.head())
print(f'\nÚltimas linhas:')
display(daily.tail())

=== Verificação Final ===
Shape: (1684, 5)
Colunas: ['date', 'sales', 'onpromotion', 'oil_price', 'is_national_holiday']
NaN por coluna:
date                   0
sales                  0
onpromotion            0
oil_price              0
is_national_holiday    0
dtype: int64

Primeiras linhas:


,date,sales,onpromotion,oil_price,is_national_holiday
0,2013-01-01,810.0,0,93.14,1
1,2013-01-02,72092.0,0,93.14,0
2,2013-01-03,52105.0,0,92.97,0
3,2013-01-04,54167.0,0,93.12,0
4,2013-01-05,77818.0,0,93.12,1



Últimas linhas:


,date,sales,onpromotion,oil_price,is_national_holiday
1679,2017-08-11,189111.0,961,48.81,1
1680,2017-08-12,182318.0,956,48.81,0
1681,2017-08-13,202354.0,986,48.81,0
1682,2017-08-14,174832.0,957,47.59,0
1683,2017-08-15,170773.0,982,47.57,0


In [14]:
# Estatísticas descritivas para RESULTADOS.md
print('=== Estatísticas Descritivas da Série ===')
stats = daily['sales'].describe()
print(stats)
print(f'\nZeros: {(daily.sales == 0).sum()}')
print(f'Variação total (último/primeiro ano): calculada na EDA')

=== Estatísticas Descritivas da Série ===
count      1684.000000
mean     128832.830166
std       63121.645563
min         810.000000
25%       69815.750000
50%      128207.500000
75%      170349.500000
max      339352.000000
Name: sales, dtype: float64

Zeros: 0
Variação total (último/primeiro ano): calculada na EDA


In [15]:
daily.to_csv(PROCESSED_DIR + 'beverages_daily.csv', index=False)
print(f'Arquivo salvo: {PROCESSED_DIR}beverages_daily.csv')
print(f'Shape final: {daily.shape}')

Arquivo salvo: ../data/processed/beverages_daily.csv
Shape final: (1684, 5)


## 1.7 Decisões tomadas nesta etapa

**Tratamento dos zeros:** mantidos na série. Todos os zeros correspondem a 1º de janeiro de cada ano, quando as lojas estão fechadas. Não representam dados ausentes, são valores reais de demanda zero. Registrado em `DECISOES.md`.

**Forward-fill no petróleo:** os NaN no `oil.csv` ocorrem em fins de semana e feriados (o mercado não opera). Propagar o último valor disponível (forward-fill) é o tratamento padrão para séries de preços financeiros nesse contexto. Se houver NaN no início da série (sem valor anterior), aplicar backward-fill pontualmente. Registrado em `DECISOES.md`.

**Feriados nacionais:** filtrado `locale == 'National'` e `transferred == False`. Feriados transferidos representam a data original de um feriado que foi oficialmente movido, o efeito real no varejo ocorre na data transferida, não na original. Portanto, excluir os registros com `transferred == True` é metodologicamente correto.